# AdOps Revenue Dashboard Builder
Run all cells top to bottom. Downloads `revenue_dashboard.html` at the end — upload to GitHub to publish.

In [ ]:
# ── Step 1: Install and authenticate ──────────────────────────────────────────
!pip install gspread google-auth --quiet
from google.colab import auth
auth.authenticate_user()
print('✓ Authenticated')

In [ ]:
# ── Step 2: Connect to Google Sheet ───────────────────────────────────────────
import gspread
from google.auth import default

SHEET_ID = '1qEnkk_RyQjpb-7MpayChxaPH1asdiwz-EJ5CkPlozNU'

creds, _ = default()
gc = gspread.authorize(creds)
ss = gc.open_by_key(SHEET_ID)
print('✓ Connected to:', ss.title)

In [ ]:
# ── Step 3: Load SF data from year tabs ───────────────────────────────────────
import pandas as pd
from datetime import date
import calendar, json, re

YEAR_TABS = ['2026', '2025', '2024', '2023', '2022', '2021', 'pre 2020 active']

def load_tab(name):
    try:
        ws = ss.worksheet(name)
        data = ws.get_all_values()
        if len(data) < 2: return pd.DataFrame()
        df = pd.DataFrame(data[1:], columns=data[0])
        df['_tab'] = name
        return df
    except Exception as e:
        print(f'  ⚠ Could not load {name}: {e}')
        return pd.DataFrame()

frames = []
for tab in YEAR_TABS:
    df = load_tab(tab)
    if len(df): 
        frames.append(df)
        print(f'  ✓ {tab}: {len(df)} rows')

sf = pd.concat(frames, ignore_index=True)
print(f'\n✓ Total: {len(sf)} rows')
print('Columns:', list(sf.columns))

In [ ]:
# ── Step 4: Load targets and lookup from revenue master sheet ─────────────────
MASTER_SHEET_ID = '1KZjhX-cuPCLi6t8vWfYPWD5P5l7gcxDk4T8IvuXbRg8'
master = gc.open_by_key(MASTER_SHEET_ID)

def master_tab(name, max_rows=None):
    ws = master.worksheet(name)
    data = ws.get_all_values()
    if max_rows: data = data[:max_rows+1]
    return pd.DataFrame(data[1:], columns=data[0])

print('Loading Lookup...')
lookup_raw = master.worksheet('Lookup').get_all_values()
lookup = pd.DataFrame(lookup_raw[1:42], columns=lookup_raw[0])  # first 41 rows only

print('Loading Reassign Revenue...')
reassign_df = master_tab('Reassign Revenue')

print('Loading Targets 2022-2025...')
t1 = master_tab('Targets 2022-2025')

print('Loading Targets 2026...')
t2 = master_tab('Targets 2026')

print('✓ All reference data loaded')

In [ ]:
def parse_gbp(v):
    if not v or str(v).strip() == '': return 0.0
    try: return float(str(v).replace('£','').replace(',','').strip())
    except: return 0.0

# ── Step 4b: Load FB Costs from master sheet ─────────────────────────────────
print('Loading FB Costs...')
fb_ws   = master.worksheet('FB costs')
fb_data = fb_ws.get_all_values()
fb_headers = fb_data[0]
print(f'  FB costs columns: {fb_headers}')
fb_rows = fb_data[1:]

# Build FB costs lookup: {(segment, package, mm_yy): forecasted_cost}
# Columns: F=MM/YY, H=Forecasted cost to end of month, I=CIN, J=Package, K=Segment
def col_idx(name):
    try: return fb_headers.index(name)
    except: return -1

# Try to find columns by header name
A_IDX = 0  # Column A = Year|Month e.g. 2025|01
H_IDX = col_idx('Forecasted cost to end of month') if col_idx('Forecasted cost to end of month') >= 0 else 7  # H
J_IDX = col_idx('Package') if col_idx('Package') >= 0 else 9   # J
K_IDX = col_idx('Segment') if col_idx('Segment') >= 0 else 10  # K

print(f'  Column indices: A(YearMonth)={A_IDX}, H(Cost)={H_IDX}, J(Package)={J_IDX}, K(Segment)={K_IDX}')

# Build lookup using column A format: 2025|01
fb_costs = {}  # {(segment, package, year_month): total_cost}
for row in fb_rows:
    if len(row) <= max(A_IDX, H_IDX, J_IDX, K_IDX):
        continue
    year_month = str(row[A_IDX]).strip()  # e.g. '2025|01'
    cost       = parse_gbp(row[H_IDX])
    package    = str(row[J_IDX]).strip()
    segment    = str(row[K_IDX]).strip()
    if not year_month or not package or not segment: continue
    key = (segment, package, year_month)
    fb_costs[key] = fb_costs.get(key, 0) + cost

print(f'  FB costs entries: {len(fb_costs)}')
# Show sample
sample = list(fb_costs.items())[:3]
for k, v in sample:
    print(f'  Sample: {k} = £{v:.2f}')

# Build margins data structure: {month: {segment: {package: {revenue, cost}}}}
MARGIN_SEGMENTS = ['NH Corporate', 'NH SME', 'EA Corporate', 'EA Key', 'EA SME', 'EA Franchise', 'Commercial', 'Overseas']
MARGIN_PRODUCTS = ['Facebook CPC', 'Facebook CPL', 'Facebook CTW', 'GDN']
MONTHS_SHORT = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
ALL_MONTHS_2021 = [f'{m} {y}' for y in range(2021,2027) for m in MONTHS_SHORT]

def month_to_mmyy(m):
    parts = m.split()
    mo = str(MONTHS_SHORT.index(parts[0]) + 1).zfill(2)
    yr = parts[1][2:]
    return f'{mo}/01/{parts[1]}'  # format: MM/01/YYYY -> need to match sheet format

# The sheet format is MM/DD/YYYY e.g. 01/01/2025 for January 2025
sample_keys = list(fb_costs.keys())[:5]
print(f'\nSample mm_yy formats: {[k[2] for k in sample_keys]}')

def month_label_to_sheet_mmyy(m):
    """Convert 'Jan 2026' to '2026|01' format used in FB costs column A"""
    MONTHS_SHORT_LOCAL = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    parts = m.split()
    mo = str(MONTHS_SHORT_LOCAL.index(parts[0]) + 1).zfill(2)
    return f'{parts[1]}|{mo}'

print(f'Test conversion: Jan 2026 -> {month_label_to_sheet_mmyy("Jan 2026")}')
print(f'Test conversion: Jun 2026 -> {month_label_to_sheet_mmyy("Jun 2026")}')


In [ ]:
# ── Step 5: Build lookup maps ──────────────────────────────────────────────────
def parse_gbp(v):
    if pd.isna(v) or str(v).strip() == '': return 0.0
    try: return float(str(v).replace('£','').replace(',','').strip())
    except: return 0.0

lookup['Package Name']  = lookup['Package Name'].str.strip()
lookup['Product Name']  = lookup['Product Name'].str.strip()
lookup['Product Group'] = lookup['Product Group'].str.strip()

PRODUCT_MAP   = dict(zip(lookup['Package Name'], lookup['Product Name']))
PRODUCT_GROUP = dict(zip(lookup['Package Name'], lookup['Product Group']))
PRODNAME_TO_PG = {}
for _, row in lookup.iterrows():
    if row['Product Name'] not in PRODNAME_TO_PG:
        PRODNAME_TO_PG[row['Product Name']] = row['Product Group']

# ── Additional mappings for new packages ──────────────────────────────────────
# UPDATE THESE when Afshan confirms the correct product group for each
EXTRA_MAPPINGS = {
    # Package Name : (Product Name, Product Group)
    # TO BE CONFIRMED:
    'Valuation Booster':                        ('Valuation Booster',        'Other Onsite'),
    'Featured Agent':                           ('Featured Agent',            'Other Onsite'),
    'Featured Development for Sale':            ('Featured Development',      'Other Onsite'),
    'Featured Development to Rent':             ('Featured Development',      'Other Onsite'),
    'Featured properties for sale':             ('Featured Properties',       'Other Onsite'),
    'Featured properties to rent':              ('Featured Properties',       'Other Onsite'),
    'Home of the Week':                         ('HOTW Boost',               'Email'),
    'In-Search Ads - Campaign':                 ('In-Search',                'In-Search'),
    'New Home Premium Listings (Subscription)': ('Premium Listings',         'Other Onsite'),
    'New Homes Bundle - Development Accelerator': ('Development Accelerator', 'Other Onsite'),
    'New Homes Bundle - Event Promoter':        ('Event Promoter',           'Other Onsite'),
    'Premium Listings (Capped prepaid credit)': ('Premium Listings',         'Other Onsite'),
    'Premium Listings (Capped subscription)':   ('Premium Listings',         'Other Onsite'),
    'Premium Listings (Pay-per-listing)':       ('Premium Listings',         'Other Onsite'),
    'Prospect Plus (CPL)':                      ('Prospect Plus',            'Remarketing'),
    'Prospect Plus (Monthly)':                  ('Prospect Plus',            'Remarketing'),
    'Weekly Featured Properties (PAYG)':        ('Weekly Featured Property', 'Other Onsite'),
    'Weekly Featured Property (Subscription)':  ('Weekly Featured Property', 'Other Onsite'),
}
for pkg, (pn, pg) in EXTRA_MAPPINGS.items():
    PRODUCT_MAP[pkg]   = pn
    PRODUCT_GROUP[pkg] = pg
    if pn not in PRODNAME_TO_PG: PRODNAME_TO_PG[pn] = pg

# Reassign overrides
REASSIGN = {}
for _, row in reassign_df.iterrows():
    cin = str(row.iloc[0]).strip()
    if cin: REASSIGN[cin] = {'product': str(row.iloc[1]).strip(), 'cost': parse_gbp(row.iloc[2])}

print(f'✓ {len(PRODUCT_MAP)} package mappings, {len(REASSIGN)} reassign overrides')

In [ ]:
# ── Step 6: Revenue calculation formula ───────────────────────────────────────
def safe_date(v):
    if pd.isna(v) or str(v).strip() == '': return None
    try: return pd.to_datetime(v, dayfirst=True).date()
    except: return None

def edate(d, months):
    m = d.month + months
    y = d.year + (m - 1) // 12
    m = (m - 1) % 12 + 1
    return date(y, m, min(d.day, calendar.monthrange(y, m)[1]))

def eomonth(d, offset):
    m = d.month + offset
    y = d.year + (m - 1) // 12
    m = (m - 1) % 12 + 1
    return date(y, m, calendar.monthrange(y, m)[1])

def calc_monthly_revenue(start, end, pkg, rate, m_date):
    if not start or not pkg: return 0
    pkg = pkg.strip()
    if pkg != 'Adreach Express' and (not rate or rate == 0): return 0
    if not end: end = date(2099, 12, 31)
    if pkg == 'Adreach Express' and (not rate or rate == 0): rate = 200

    month_start = date(m_date.year, m_date.month, 1)
    month_end   = eomonth(m_date, 0)
    days_in_month = month_end.day

    def overlap(s, e): return max(0, (min(e, month_end) - max(s, month_start)).days + 1)

    PRO_RATA = {'Facebook Lead Form (CPC)','Facebook Lead Form (CPL)','Facebook CTW',
                'In-Search Ads - Fixed Time Campaign','Leaderboard - Fixed Time Campaign',
                'Google Display Network Ads','In-Search Ads - Packaged - Monthly'}
    MONTHLY_PRO_RATA  = {'Direct Campaign - Monthly'}
    START_TRIGGER     = {'Direct Campaign - Upfront','In-Search Ads - Packaged - Upfront',
                         'Email Campaign','HOTW Boost','In-Search Ads - Custom - Upfront'}
    MONTH_AFTER_TRIG  = {'Email Campaign - Off Plan Package'}
    CATCHUP_SUB       = {'Property Sponsorship','In-Search Ads','Direct Campaign',
                         'AdReach Express Premium Subscription','AdReach Express Subscription',
                         'Area Sponsorship','Area Sponsorship (Zone 2)','Area Sponsorship (Zone 3)',
                         'Area Sponsorship on PrimeLocation','Audience Connect - ZVT',
                         'Audience Connect - Chatbot','Audience Connect - chatbox'}
    DELAYED_PRO_RATA  = {'Remarketing: Bespoke Subscription','AdReach - Always On',
                         'Leaderboard','Bespoke Subscription','Adreach Express'}
    MONTHLY_FIXED     = {'Property Alerts'}

    if pkg in PRO_RATA:
        contract_days = max(1, (end - start).days + 1)
        return (rate / contract_days) * overlap(start, end)

    elif pkg in MONTHLY_PRO_RATA:
        return (rate / days_in_month) * overlap(start, end)

    elif pkg in START_TRIGGER:
        return rate if month_start <= start <= month_end else 0

    elif pkg in MONTH_AFTER_TRIG:
        trigger = edate(start, 1)
        return rate if month_start <= trigger <= month_end else 0

    elif pkg in CATCHUP_SUB:
        next_month  = edate(start, 1)
        eomonth_s   = eomonth(start, 0)
        if month_start.year == next_month.year and month_start.month == next_month.month:
            partial = (rate / eomonth_s.day) * ((eomonth_s - start).days + 1)
            return partial + rate
        elif month_start > next_month and month_start <= end:
            return rate
        else:
            return 0

    elif pkg in DELAYED_PRO_RATA:
        delay_start = edate(start, 1)
        cutoff = eomonth(edate(start, 1), -1)
        if month_start < cutoff: return 0
        return (rate / days_in_month) * overlap(delay_start, end)

    elif pkg in MONTHLY_FIXED:
        return rate if overlap(start, end) > 0 else 0

    else:
        return 0  # unknown package type = 0

print('✓ Revenue formula ready')

In [ ]:
# ── Step 7: Process contracts ──────────────────────────────────────────────────
MONTHS_SHORT  = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
ALL_MONTHS    = [f'{m} {y}' for y in range(2021,2027) for m in MONTHS_SHORT]
EXCLUDE_CH    = {'Advertiser','Media Agency','Network Partner - Broker','Third Party','Third party'}
ADSALES_CH    = {'Advertiser','Third Party','Third party'}  # kept separately for AdSales margins row
NH_CH         = {'NH Corporate','NH SME'}
EA_CH         = {'EA Key','EA Corporate','EA SME','EA Franchise','Commercial','Overseas'}

# Find column names
def find_col(df, *candidates):
    for c in candidates:
        if c in df.columns: return c
    return None

COL_CIN     = find_col(sf, 'Contract Item: Contract Item Name','Contract Item Name')
COL_STATUS  = find_col(sf, 'Status')
COL_CHANNEL = find_col(sf, 'Sub-Channel')
COL_PKG     = find_col(sf, 'Package Name')
COL_PRODUCT = find_col(sf, 'Product')
COL_RATE    = find_col(sf, 'Rate','New Rate')
COL_START   = find_col(sf, 'Start Date')
COL_END     = find_col(sf, 'End Date')
COL_OPP     = find_col(sf, 'Opportunity')

print(f'CIN={COL_CIN}, Status={COL_STATUS}, Channel={COL_CHANNEL}, Pkg={COL_PKG}, Rate={COL_RATE}')

contracts = []
contracts_adsales = []
skipped   = 0
for idx, row in sf.iterrows():
    cin     = str(row[COL_CIN]).strip()     if COL_CIN     else ''
    status  = str(row[COL_STATUS]).strip()  if COL_STATUS  else ''
    channel = str(row[COL_CHANNEL]).strip() if COL_CHANNEL else ''
    pkg     = str(row[COL_PKG]).strip()     if COL_PKG     else ''
    rate    = parse_gbp(row[COL_RATE])      if COL_RATE    else 0
    opp     = str(row[COL_OPP]).strip().lower() if COL_OPP else ''
    start   = safe_date(row[COL_START])     if COL_START   else None
    end     = safe_date(row[COL_END])       if COL_END     else None

    is_adsales = channel in ADSALES_CH
    if (channel in EXCLUDE_CH and not is_adsales) or not start or not pkg:
        skipped += 1; continue

    is_draft = (status == 'Draft')
    if not is_draft and status not in ('Active','Expired','Cancelled'):
        skipped += 1; continue

    # Product name and group
    if cin in REASSIGN:
        prod_name  = REASSIGN[cin]['product']
        prod_group = PRODNAME_TO_PG.get(prod_name, 'Other Onsite')
        rate       = REASSIGN[cin]['cost'] if REASSIGN[cin]['cost'] > 0 else rate
    else:
        prod_name  = PRODUCT_MAP.get(pkg, pkg)
        prod_group = PRODUCT_GROUP.get(pkg, 'Other')

    # Adreach Express guild gold rule: rate=0 + opp contains 'guild gold' → rate=200
    if prod_name == 'Adreach Express' and rate == 0 and 'guild gold' in opp:
        rate = 200

    revenue = {}
    for m in ALL_MONTHS:
        yr2 = int(m.split()[1])
        mo2 = MONTHS_SHORT.index(m.split()[0]) + 1
        rev = calc_monthly_revenue(start, end, pkg, rate, date(yr2, mo2, 1))
        if rev > 0: revenue[m] = rev

    contracts.append({
        'channel':channel, 'productName':prod_name,
        'productGroup':prod_group, 'isDraft':is_draft, 'revenue':revenue
    })

print(f'✓ Processed {len(contracts)} contracts, skipped {skipped}')
# Quick sanity check
jan26 = sum(c['revenue'].get('Jan 2026',0) for c in contracts if not c['isDraft'])
print(f'  Jan 2026 total revenue: £{jan26:,.0f}')

In [ ]:
# ── Step 8: Build targets ──────────────────────────────────────────────────────
CHAN_MAP  = {'Commercials':'Commercial','Franchise':'EA Franchise'}
PG_MAP_T1 = {'Onsite':'Other Onsite','Email':'Email','Remarketing':'Remarketing','Programmatic':'In-Search'}
PG_MAP_T2 = {'In-Search':'In-Search','Other Onsite':'Other Onsite','Remarketing':'Remarketing','Email':'Email'}
VALID_CH1 = {'EA Corporate','Commercials','Overseas','Franchise','EA Key','EA SME','NH SME','NH Corporate'}
VALID_CH2 = {'EA Corporate','EA SME','Franchise','NH Corporate','NH SME','Commercial','Overseas'}

tgt_rows = []

# 2022-2025
t1_mcols = [c for c in t1.columns if any(y in c for y in ['2022','2023','2024','2025'])]
for _, row in t1[t1['Sales Channel'].isin(VALID_CH1)].iterrows():
    sc = CHAN_MAP.get(row['Sales Channel'], row['Sales Channel'])
    pg = PG_MAP_T1.get(str(row.get('Product Group','')).strip())
    if not pg: continue
    for mc in t1_mcols:
        tgt_rows.append({'channel':sc,'pg':pg,'month':mc.strip(),'target':parse_gbp(row[mc])})

# 2026
t2_mcols = [c for c in t2.columns if '2026' in c]
t2_ch_col = t2.columns[0]
for _, row in t2.iloc[:28].iterrows():
    sc = str(row[t2_ch_col]).strip()
    if sc not in VALID_CH2: continue
    sc = CHAN_MAP.get(sc, sc)
    pg = PG_MAP_T2.get(str(row.get('Product','')).strip())
    if not pg: continue
    for mc in t2_mcols:
        tgt_rows.append({'channel':sc,'pg':pg,'month':mc.strip(),'target':parse_gbp(row[mc])})

targets = pd.DataFrame(tgt_rows)
print(f'✓ Built {len(targets)} target rows')

In [ ]:
# ── Step 9: Aggregate dashboard data (fast pandas approach) ──────────────────
import pandas as pd
import numpy as np

PGS_PRE2026 = ['Email','Other Onsite','Remarketing']
PGS_2026    = ['Email','In-Search','Other Onsite','Remarketing']
def pgs_for(m): return PGS_2026 if int(m.split()[1])==2026 else PGS_PRE2026

NH_CH  = {'NH Corporate','NH SME'}
EA_CH  = {'EA Key','EA Corporate','EA SME','EA Franchise','Commercial','Overseas'}
NH_CHANS = list(NH_CH)
EA_CHANS = list(EA_CH)

# ── Build a flat DataFrame from contracts ──────────────────────────────────────
rows = []
for c in contracts:
    for m, v in c['revenue'].items():
        rows.append({
            'month':        m,
            'channel':      c['channel'],
            'productName':  c['productName'],
            'productGroup': c['productGroup'],
            'isDraft':      c['isDraft'],
            'revenue':      v
        })

flat = pd.DataFrame(rows)
flat['isNH']    = flat['channel'].isin(NH_CH)
flat['isEA']    = flat['channel'].isin(EA_CH)
flat_act  = flat[~flat['isDraft']].copy()
flat_drft = flat[flat['isDraft']].copy()

channels      = sorted(flat_act['channel'].unique().tolist())
product_names = sorted(flat_act['productName'].unique().tolist())

print(f'✓ Flat table: {len(flat_act):,} actual rows, {len(flat_drft):,} draft rows')

# ── Pre-aggregate all needed groupbys ─────────────────────────────────────────
# Total by month
grp_total    = flat_act.groupby('month')['revenue'].sum()
grp_draft    = flat_drft.groupby('month')['revenue'].sum()

# By channel
grp_ch       = flat_act.groupby(['month','channel'])['revenue'].sum()
grp_nh       = flat_act[flat_act['isNH']].groupby('month')['revenue'].sum()
grp_ea       = flat_act[flat_act['isEA']].groupby('month')['revenue'].sum()

# By product group (year-aware handled later)
grp_pg       = flat_act.groupby(['month','productGroup'])['revenue'].sum()
grp_nh_pg    = flat_act[flat_act['isNH']].groupby(['month','productGroup'])['revenue'].sum()
grp_ea_pg    = flat_act[flat_act['isEA']].groupby(['month','productGroup'])['revenue'].sum()

# By product name
grp_pn       = flat_act.groupby(['month','productName'])['revenue'].sum()

# By channel × product name (for table view filtering)
grp_ch_pn    = flat_act.groupby(['month','channel','productName'])['revenue'].sum()
grp_nh_pn    = flat_act[flat_act['isNH']].groupby(['month','productName'])['revenue'].sum()
grp_ea_pn    = flat_act[flat_act['isEA']].groupby(['month','productName'])['revenue'].sum()

# Draft by channel and PG
grp_drft_ch  = flat_drft.groupby(['month','channel'])['revenue'].sum()
grp_drft_nh  = flat_drft[flat_drft['isNH']].groupby('month')['revenue'].sum()
grp_drft_ea  = flat_drft[flat_drft['isEA']].groupby('month')['revenue'].sum()
grp_drft_pg  = flat_drft.groupby(['month','productGroup'])['revenue'].sum()
grp_drft_pn    = flat_drft.groupby(['month','productName'])['revenue'].sum()
grp_drft_nh_pn = flat_drft[flat_drft['isNH']].groupby(['month','productName'])['revenue'].sum()

print('✓ All groupbys computed')

# ── Load targets ───────────────────────────────────────────────────────────────
def sum_target(m, chans=None, pg=None):
    mask = targets['month']==m
    if chans: mask &= targets['channel'].isin(chans)
    if pg:    mask &= (targets['pg']==pg)
    return round(float(targets[mask]['target'].sum()))

# ── Build monthly dicts ────────────────────────────────────────────────────────
ALL_MONTHS = [f'{m} {y}' for y in range(2021,2027)
              for m in ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']]

def g(grp, *keys):
    try: return round(float(grp.loc[keys] if len(keys)>1 else grp.loc[keys[0]]))
    except: return 0

monthly={};monthly_by_channel={};monthly_by_product_group={}
monthly_by_product_name={};monthly_draft={};monthly_by_channel_draft={}
monthly_ch_pg_actual={};monthly_ch_pg_target={};monthly_ch_product_actual={}
monthly_draft_by_product={};monthly_nh_draft_by_product={}

for m in ALL_MONTHS:
    yr   = int(m.split()[1])
    pgs  = pgs_for(m)

    monthly[m] = {'actual': g(grp_total,m), 'target': sum_target(m)}

    # By channel
    monthly_by_channel[m] = {ch: {'actual': g(grp_ch,m,ch), 'target': sum_target(m,[ch])} for ch in channels}
    monthly_by_channel[m]['__NH__'] = {'actual': g(grp_nh,m), 'target': sum_target(m,NH_CHANS)}
    monthly_by_channel[m]['__EA__'] = {'actual': g(grp_ea,m), 'target': sum_target(m,EA_CHANS)}

    # By product group (pre-2026: merge In-Search into Other Onsite)
    monthly_by_product_group[m] = {}
    for pg in pgs:
        if pg == 'Other Onsite' and yr < 2026:
            act = g(grp_pg,m,'Other Onsite') + g(grp_pg,m,'In-Search')
        else:
            act = g(grp_pg,m,pg)
        monthly_by_product_group[m][pg] = {'actual': act, 'target': sum_target(m,pg=pg)}

    # By product name
    monthly_by_product_name[m] = {pn: {'actual': g(grp_pn,m,pn), 'target':0} for pn in product_names}

    # Draft
    monthly_draft[m] = g(grp_draft,m)
    monthly_by_channel_draft[m] = {ch: g(grp_drft_ch,m,ch) for ch in channels}
    monthly_by_channel_draft[m]['__NH__'] = g(grp_drft_nh,m)
    monthly_by_channel_draft[m]['__EA__'] = g(grp_drft_ea,m)
    monthly_by_channel_draft[m]['__by_pg__'] = {pg: g(grp_drft_pg,m,pg) for pg in pgs}
    # NH draft by channel
    for ch in channels:
        monthly_by_channel_draft[m][ch] = g(grp_drft_ch,m,ch)

    # Draft by product name (all channels, and NH-only)
    monthly_draft_by_product[m]    = {pn: g(grp_drft_pn,m,pn) for pn in product_names}
    monthly_nh_draft_by_product[m] = {pn: g(grp_drft_nh_pn,m,pn) for pn in product_names}

    # Channel × PG actual and target
    monthly_ch_pg_actual[m] = {
        '__NH__': {pg: (g(grp_nh_pg,m,'Other Onsite')+g(grp_nh_pg,m,'In-Search') if pg=='Other Onsite' and yr<2026 else g(grp_nh_pg,m,pg)) for pg in pgs},
        '__EA__': {pg: (g(grp_ea_pg,m,'Other Onsite')+g(grp_ea_pg,m,'In-Search') if pg=='Other Onsite' and yr<2026 else g(grp_ea_pg,m,pg)) for pg in pgs},
    }
    monthly_ch_pg_target[m] = {
        '__NH__': {pg: sum_target(m,NH_CHANS,pg) for pg in pgs},
        '__EA__': {pg: sum_target(m,EA_CHANS,pg) for pg in pgs},
    }

    # Channel × product name actual (for table view)
    monthly_ch_product_actual[m] = {
        '__NH__': {pn: g(grp_nh_pn,m,pn) for pn in product_names},
        '__EA__': {pn: g(grp_ea_pn,m,pn) for pn in product_names},
    }
    for ch in channels:
        monthly_ch_product_actual[m][ch] = {pn: g(grp_ch_pn,m,ch,pn) for pn in product_names}

# ── Annual rollups ─────────────────────────────────────────────────────────────
by_channel={};by_product_group={};by_product_name={};running={}
for yr in range(2021,2027):
    ys=str(yr); yms=[m for m in ALL_MONTHS if m.endswith(ys)]; pgs=pgs_for(yms[0])
    by_channel[ys]={ch:{'actual':sum(monthly_by_channel[m].get(ch,{}).get('actual',0) for m in yms),
                         'target':sum(monthly_by_channel[m].get(ch,{}).get('target',0) for m in yms)}
                    for ch in channels+['__NH__','__EA__']}
    by_product_group[ys]={pg:{'actual':sum(monthly_by_product_group[m].get(pg,{}).get('actual',0) for m in yms),
                               'target':sum(monthly_by_product_group[m].get(pg,{}).get('target',0) for m in yms)} for pg in pgs}
    by_product_name[ys]={pn:{'actual':sum(monthly_by_product_name[m].get(pn,{}).get('actual',0) for m in yms),'target':0} for pn in product_names}
    cumA=cumT=cumNHA=cumNHT=cumEAA=cumEAT=0
    ra=[];rt=[];rnha=[];rnht=[];reaa=[];reat=[];rms=[]
    for m in yms:
        cumA+=monthly[m]['actual'];cumT+=monthly[m]['target']
        cumNHA+=monthly_by_channel[m]['__NH__']['actual'];cumNHT+=monthly_by_channel[m]['__NH__']['target']
        cumEAA+=monthly_by_channel[m]['__EA__']['actual'];cumEAT+=monthly_by_channel[m]['__EA__']['target']
        ra.append(cumA);rt.append(cumT);rnha.append(cumNHA);rnht.append(cumNHT)
        reaa.append(cumEAA);reat.append(cumEAT);rms.append(m.split()[0])
    dpg={pg:sum(monthly_by_channel_draft[m].get('__by_pg__',{}).get(pg,0) for m in yms) for pg in pgs}
    running[ys]={'actual':ra,'target':rt,'nh_actual':rnha,'nh_target':rnht,'ea_actual':reaa,'ea_target':reat,
                 'months':rms,'product_groups':pgs,'draft_by_pg':dpg}
    if yr==2026:
        cwd=0;awd=[]
        for m in yms: cwd+=monthly[m]['actual']+(monthly_draft.get(m,0));awd.append(cwd)
        running['2026']['actual_with_draft']=awd

# ── Build AdSales product×month aggregation from contracts_adsales ───────────
monthly_adsales_product = {}
for m in ALL_MONTHS:
    monthly_adsales_product[m] = {}
    for c in contracts_adsales:
        if c['isDraft']: continue
        rev = c['revenue'].get(m, 0)
        if rev > 0:
            pn = c['productName']
            monthly_adsales_product[m][pn] = monthly_adsales_product[m].get(pn, 0) + rev

# ── Build margins data ────────────────────────────────────────────────────────
MARGIN_SEGMENTS_NH     = ['NH Corporate', 'NH SME']
MARGIN_SEGMENTS_EA_RAW = ['EA Key', 'EA SME', 'EA Franchise', 'Commercial', 'Overseas']  # EA Corporate now separate
MARGIN_PRODUCTS_NH        = ['Facebook CPC', 'Facebook CPL', 'Facebook CTW', 'GDN']
MARGIN_PRODUCTS_EA        = ['Bespoke Subscription', 'GDN', 'Adreach Express', 'Audience Connect']
MARGIN_PRODUCTS_EA_CORP   = ['Facebook CPC', 'Facebook CPL', 'Facebook CTW', 'Bespoke Subscription']
MARGIN_PRODUCTS_ADSALES   = ['Facebook CPC', 'Facebook CPL', 'Facebook CTW']

margins = {}  # {month: {segment: {package: {revenue, cost}}}}
for m in ALL_MONTHS:
    sheet_mmyy = month_label_to_sheet_mmyy(m)
    margins[m] = {}

    # NH segments — Facebook products
    for seg in MARGIN_SEGMENTS_NH:
        margins[m][seg] = {}
        for pkg in MARGIN_PRODUCTS_NH:
            rev = monthly_ch_product_actual.get(m, {}).get(seg, {}).get(pkg, 0)
            cost = fb_costs.get((seg, pkg, sheet_mmyy), 0)
            if rev > 0 or cost > 0:
                margins[m][seg][pkg] = {'revenue': round(rev, 2), 'cost': round(cost, 2)}

    # EA Corporate — separate row, its own product set
    margins[m]['EA Corporate'] = {}
    for pkg in MARGIN_PRODUCTS_EA_CORP:
        rev = monthly_ch_product_actual.get(m, {}).get('EA Corporate', {}).get(pkg, 0)
        cost = fb_costs.get(('EA Corporate', pkg, sheet_mmyy), 0)
        if rev > 0 or cost > 0:
            margins[m]['EA Corporate'][pkg] = {'revenue': round(rev, 2), 'cost': round(cost, 2)}

    # EA combined (Key, SME, Franchise, Commercial, Overseas) — AdReach/GDN products
    margins[m]['EA'] = {}
    for pkg in MARGIN_PRODUCTS_EA:
        rev  = sum(monthly_ch_product_actual.get(m, {}).get(seg, {}).get(pkg, 0) for seg in MARGIN_SEGMENTS_EA_RAW)
        cost = sum(fb_costs.get((seg, pkg, sheet_mmyy), 0) for seg in MARGIN_SEGMENTS_EA_RAW)
        if rev > 0 or cost > 0:
            margins[m]['EA'][pkg] = {'revenue': round(rev, 2), 'cost': round(cost, 2)}

    # AdSales — Advertiser + Third Party channels combined
    margins[m]['AdSales'] = {}
    for pkg in MARGIN_PRODUCTS_ADSALES:
        rev  = monthly_adsales_product.get(m, {}).get(pkg, 0)
        cost = fb_costs.get(('AdSales', pkg, sheet_mmyy), 0)
        if rev > 0 or cost > 0:
            margins[m]['AdSales'][pkg] = {'revenue': round(rev, 2), 'cost': round(cost, 2)}

# Sanity check
test_m = 'Jun 2026'
print(f'\nMargins sample for {test_m}:')
for seg in ['NH Corporate','NH SME']:
    for pkg in ['Facebook CPC']:
        v = margins.get(test_m,{}).get(seg,{}).get(pkg,{})
        if v: print(f'  {seg} {pkg}: rev=£{v["revenue"]:,.2f}, cost=£{v["cost"]:,.2f}')

DATA={'monthly':monthly,'by_channel':by_channel,'by_product_group':by_product_group,
      'by_product_name':by_product_name,'running':running,
      'monthly_by_channel':monthly_by_channel,'monthly_by_product_group':monthly_by_product_group,
      'monthly_by_product_name':monthly_by_product_name,'monthly_draft':monthly_draft,
      'monthly_by_channel_draft':monthly_by_channel_draft,'monthly_ch_pg_actual':monthly_ch_pg_actual,
      'monthly_ch_pg_target':monthly_ch_pg_target,'monthly_ch_product_actual':monthly_ch_product_actual,
      'monthly_draft_by_product':monthly_draft_by_product,'monthly_nh_draft_by_product':monthly_nh_draft_by_product,
      'channels':channels,'product_names':product_names,
      'nh_channels':NH_CHANS,'ea_channels':EA_CHANS,'pgs_pre2026':PGS_PRE2026,'pgs_2026':PGS_2026,
      'margins':margins,
      'margin_nh_segments':MARGIN_SEGMENTS_NH,
      'margin_nh_products':MARGIN_PRODUCTS_NH,
      'margin_ea_products':MARGIN_PRODUCTS_EA,
      'margin_ea_corp_products':MARGIN_PRODUCTS_EA_CORP,
      'margin_adsales_products':MARGIN_PRODUCTS_ADSALES}

print(f'✓ Data built')
print(f'  Jan 2026 total:  £{monthly["Jan 2026"]["actual"]:,}')
print(f'  NH Jan 2026:     £{monthly_by_channel["Jan 2026"]["__NH__"]["actual"]:,}')
print(f'  EA Jan 2026:     £{monthly_by_channel["Jan 2026"]["__EA__"]["actual"]:,}')


In [ ]:
# ── Step 10: Build and download dashboard ─────────────────────────────────────
import re, urllib.request
from google.colab import files

GITHUB_RAW_URL = 'https://raw.githubusercontent.com/afshanqayyum-lgtm/revenue_dashboard/main/revenue_dashboard.html'

with urllib.request.urlopen(GITHUB_RAW_URL) as r:
    template = r.read().decode('utf-8')
print('✓ Template fetched from GitHub')

updated = re.sub(r'const DATA = \{.*?\};', f'const DATA = {json.dumps(DATA)};', template, flags=re.DOTALL)

with open('revenue_dashboard.html','w') as f:
    f.write(updated)

files.download('revenue_dashboard.html')
print('✓ Done — upload revenue_dashboard.html to GitHub to publish')